In [1]:
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY1")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")

llm=ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0.8,
    max_tokens=20
)

In [7]:
from langchain.chains import create_sql_query_chain
from langchain_community.utilities import SQLDatabase
#create_sql_query_chain
db_user='root'
db_password='123456'
db_host='localhost'
db_port=3306
db_database='rfm_db'
db=SQLDatabase.from_uri(f'mysql+pymysql://{db_user}:{db_password}@{db_host}:{db_port}/{db_database}')

print(db.dialect)
print(db.get_usable_table_names())

query='select count(*) from rfm_table'
res=db.run(query)
print(res)

mysql
['rfm_table']
[(93,)]


create_sql_query_chain的使用

In [8]:
chain=create_sql_query_chain(llm,db)
res=chain.invoke({'question':'该表中有多少条数据'})
print(res)

SQLQuery: SELECT COUNT(*) AS `total_records` FROM rfm_table;


create_stuff_documents_chain(了解)

In [9]:
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document

# 定义提示词模板
prompt = PromptTemplate.from_template("""
如下文档{docs}中说，香蕉是什么颜色的？
""")

# 创建链
llm = ChatOpenAI(model="gpt-4o-mini")
chain = create_stuff_documents_chain(llm, prompt, document_variable_name="docs")

# 文档输入
docs = [
    Document(
        page_content="苹果，学名Malus pumila Mill.，别称西洋苹果、柰，属于蔷薇科苹果属的植物。苹果是全球最广泛种植和销售的水果之一，具有悠久的栽培历史和广泛的分布范围。苹果的原始种群主要起源于中亚的天山山脉附近，尤其是现代哈萨克斯坦的阿拉木图地区，提供了所有现代苹果品种的基因库。苹果通过早期的贸易路线，如丝绸之路，从中亚向外扩散到全球各地。"
    ),
    Document(
        page_content="香蕉是白色的水果，主要产自热带地区。"

    ),
    Document(
        page_content="蓝莓是蓝色的浆果，含有抗氧化物质。"

    )
]
# 执行摘要
chain.invoke({"docs": docs})

'文中提到香蕉是白色的水果。'